In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import math
from scipy.ndimage import zoom
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score, confusion_matrix, cohen_kappa_score
from copy import deepcopy
from tqdm import tqdm

# Note: Ensure you have `mamba_ssm` and `open_clip` installed in your Kaggle environment
try:
    from mamba_ssm import Mamba
except ImportError:
    print("Please install mamba_ssm: !pip install mamba-ssm")

# ==========================================
# STAGE 1: RGB Proxy & Pseudo-Label Generation
# ==========================================


def get_indian_pines_rgb(raw_hsi_cube):
    """
    Extracts RGB bands from the raw Indian Pines (AVIRIS) data cube.
    Commonly, bands 43, 21, and 11 approximate Red, Green, and Blue.
    """
    # Assuming shape [H, W, Spectral]
    r_band = raw_hsi_cube[:, :, 43]
    g_band = raw_hsi_cube[:, :, 21]
    b_band = raw_hsi_cube[:, :, 11]

    rgb_image = np.stack([r_band, g_band, b_band], axis=-1)

    # Normalize to [0, 1] for CLIP/SegEarth-OV
    rgb_image = (rgb_image - rgb_image.min()) / (
        rgb_image.max() - rgb_image.min() + 1e-8
    )
    return rgb_image


def generate_pseudo_labels(rgb_image, label_list, segov_model, device="cuda"):
    """
    Implements the Resolution Scaling (RS) strategy for SegEarth-OV zero-shot inference.
    """
    scale_list = [1, 2]
    slide_crop = 224
    slide_stride = 112

    H, W, _ = rgb_image.shape
    num_classes = len(label_list)
    logits_seg = np.zeros((H, W, num_classes))

    for scale in scale_list:
        # Scale the image
        img_resized = zoom(rgb_image, (scale, scale, 1), order=3)

        # Note: Replace `perform_segearth` with your actual SegEarth inference call
        # pre_seg_, logits_seg_ = perform_segearth(segov_model, img_resized)

        # MOCKUP: Simulating SegEarth-OV output for code completeness
        logits_seg_ = np.random.randn(
            img_resized.shape[0], img_resized.shape[1], num_classes
        )

        # Scale back the logits
        logits_seg_ = zoom(
            logits_seg_.astype(np.float32), (1 / scale, 1 / scale, 1), order=3
        )
        logits_seg += logits_seg_ / len(scale_list)

    pre_seg = logits_seg.argmax(axis=-1)
    return pre_seg, logits_seg


# ==========================================
# STAGE 2: Noisy Label Learning with MambaHSI
# ==========================================


class SpeMamba(nn.Module):
    def __init__(self, channels, token_num=8, use_residual=True, group_num=4):
        super().__init__()
        self.token_num = token_num
        self.use_residual = use_residual
        self.group_channel_num = math.ceil(channels / token_num)
        self.channel_num = self.token_num * self.group_channel_num

        self.mamba = Mamba(
            d_model=self.group_channel_num,
            d_state=16,
            d_conv=4,
            expand=2,
        )
        self.proj = nn.Sequential(nn.GroupNorm(group_num, self.channel_num), nn.SiLU())

    def padding_feature(self, x):
        B, C, H, W = x.shape
        if C < self.channel_num:
            pad_c = self.channel_num - C
            pad_features = torch.zeros((B, pad_c, H, W)).to(x.device)
            return torch.cat([x, pad_features], dim=1)
        return x

    def forward(self, x):
        x_pad = self.padding_feature(x)
        x_pad = x_pad.permute(0, 2, 3, 1).contiguous()
        B, H, W, C_pad = x_pad.shape
        x_flat = x_pad.view(B * H * W, self.token_num, self.group_channel_num)
        x_flat = self.mamba(x_flat)
        x_recon = x_flat.view(B, H, W, C_pad).permute(0, 3, 1, 2).contiguous()
        x_proj = self.proj(x_recon)
        return x + x_proj if self.use_residual else x_proj


class SpaMamba(nn.Module):
    def __init__(self, channels, use_residual=True, group_num=4, use_proj=True):
        super().__init__()
        self.use_residual = use_residual
        self.use_proj = use_proj
        self.mamba = Mamba(
            d_model=channels,
            d_state=16,
            d_conv=4,
            expand=2,
        )
        if self.use_proj:
            self.proj = nn.Sequential(nn.GroupNorm(group_num, channels), nn.SiLU())

    def forward(self, x):
        x_re = x.permute(0, 2, 3, 1).contiguous()
        B, H, W, C = x_re.shape
        x_flat = x_re.view(1, -1, C)
        x_flat = self.mamba(x_flat)
        x_recon = x_flat.view(B, H, W, C).permute(0, 3, 1, 2).contiguous()
        if self.use_proj:
            x_recon = self.proj(x_recon)
        return x_recon + x if self.use_residual else x_recon


class BothMamba(nn.Module):
    def __init__(self, channels, token_num, use_residual, group_num=4, use_att=True):
        super().__init__()
        self.use_att = use_att
        self.use_residual = use_residual
        if self.use_att:
            self.weights = nn.Parameter(torch.ones(2) / 2)
            self.softmax = nn.Softmax(dim=0)

        self.spa_mamba = SpaMamba(
            channels, use_residual=use_residual, group_num=group_num
        )
        self.spe_mamba = SpeMamba(
            channels,
            token_num=token_num,
            use_residual=use_residual,
            group_num=group_num,
        )

    def forward(self, x):
        spa_x = self.spa_mamba(x)
        spe_x = self.spe_mamba(x)
        if self.use_att:
            weights = self.softmax(self.weights)
            fusion_x = spa_x * weights[0] + spe_x * weights[1]
        else:
            fusion_x = spa_x + spe_x
        return fusion_x + x if self.use_residual else fusion_x


class MambaHSI(nn.Module):
    def __init__(
        self,
        in_channels=50,
        hidden_dim=64,
        num_classes=16,
        use_residual=True,
        token_num=4,
        group_num=4,
    ):
        super().__init__()

        # Adapting to PCA 50 input
        self.patch_embedding = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=hidden_dim,
                kernel_size=1,
                stride=1,
                padding=0,
            ),
            nn.GroupNorm(group_num, hidden_dim),
            nn.SiLU(),
        )

        self.mamba = nn.Sequential(
            BothMamba(
                channels=hidden_dim,
                token_num=token_num,
                use_residual=use_residual,
                group_num=group_num,
            ),
            nn.AvgPool2d(kernel_size=2, stride=2, padding=0),
            BothMamba(
                channels=hidden_dim,
                token_num=token_num,
                use_residual=use_residual,
                group_num=group_num,
            ),
            nn.AvgPool2d(kernel_size=2, stride=2, padding=0),
            BothMamba(
                channels=hidden_dim,
                token_num=token_num,
                use_residual=use_residual,
                group_num=group_num,
            ),
        )

        self.upsample = nn.Sequential(
            nn.Upsample(scale_factor=4, mode="bilinear", align_corners=True),
            nn.Conv2d(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.GroupNorm(group_num, hidden_dim),
            nn.SiLU(),
        )

        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=hidden_dim,
            kernel_size=3,
            stride=1,
            padding=1,
        )

        self.cls_head = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_dim * 2,
                out_channels=hidden_dim,
                kernel_size=1,
                stride=1,
                padding=0,
            ),
            nn.GroupNorm(group_num, hidden_dim),
            nn.SiLU(),
            nn.Conv2d(
                in_channels=hidden_dim,
                out_channels=num_classes,
                kernel_size=1,
                stride=1,
                padding=0,
            ),
        )

    def forward(self, x):
        # Handle the [B, 1, 50, 9, 9] input format by squeezing the singleton channel dimension
        if x.dim() == 5 and x.size(1) == 1:
            x = x.squeeze(1)  # Output: [B, 50, 9, 9]

        inp = x
        x = self.patch_embedding(x)
        x = self.mamba(x)
        x = self.upsample(x)

        # Ensure sizes match before concatenation
        if x.shape[2:] != inp.shape[2:]:
            x = nn.functional.interpolate(x, size=inp.shape[2:])

        x = torch.cat((x, self.conv(inp)), dim=1)

        # Global Average Pooling for patch-based classification
        x = F.adaptive_avg_pool2d(x, 1)
        logits = self.cls_head(x).squeeze(-1).squeeze(-1)  # [B, num_classes]

        return logits


# ==========================================
# LABEL REFINEMENT & BvSB CONFIDENCE (GMM)
# ==========================================


def calculate_bvsb_confidence(logits):
    """Calculates Best versus Second Best (BvSB) margin."""
    logits_sorted = np.sort(logits, axis=-1)
    bvsb = logits_sorted[..., -1] - logits_sorted[..., -2]
    return bvsb


def separate_confident_hard_labels(
    pca_features, pseudo_labels, bvsb_scores, num_classes
):
    """
    Fits GMMs on features to dynamically re-weight soft labels for NLL.
    """
    clean_probs = np.zeros((pca_features.shape[0], num_classes))

    # Simple placeholder logic to match the paper's intent.
    # Group samples by pseudo label class
    for c in range(num_classes):
        class_idx = np.where(pseudo_labels == c)[0]
        if len(class_idx) < 5:
            continue

        class_features = pca_features[class_idx]
        class_bvsb = bvsb_scores[class_idx]

        # Fit GMM to differentiate high/low confidence within the class
        try:
            gmm = GaussianMixture(n_components=2, random_state=42).fit(
                class_bvsb.reshape(-1, 1)
            )
            confident_cluster = np.argmax(gmm.means_)
            labels = gmm.predict(class_bvsb.reshape(-1, 1))

            # Create soft targets
            clean_probs[class_idx[labels == confident_cluster], c] = 0.9  # Confident
            clean_probs[class_idx[labels != confident_cluster], c] = 0.5  # Hard/Noisy
        except ValueError:
            clean_probs[class_idx, c] = 0.8  # Fallback if too few samples

    return clean_probs


# ==========================================
# TRAINING EXECUTION (Tying it to DataLoaders)
# ==========================================


def train_special_pipeline(
    train_loader, test_loader, raw_hsi_cube, label_list, epochs=100, device="cuda"
):
    num_classes = len(label_list)

    print("--- Stage 1: Zero-Shot Pseudo Label Generation ---")
    rgb_proxy = get_indian_pines_rgb(raw_hsi_cube)
    # Placeholder: Assuming you instantiate SegEarth-OV elsewhere in your environment
    segov_model = None
    pseudo_labels, pseudo_logits = generate_pseudo_labels(
        rgb_proxy, label_list, segov_model, device
    )
    bvsb_scores = calculate_bvsb_confidence(pseudo_logits)

    print("--- Stage 2: NLL with MambaHSI ---")
    model = MambaHSI(in_channels=50, num_classes=num_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-5
    )

    # Optional: You would map the pseudo_labels & bvsb_scores back to the train_loader patches here.
    # For integration, the standard cross-entropy loop applies.

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_idx, (data, _) in enumerate(
            train_loader
        ):  # Ignore true labels during training
            data = data.to(device)

            # In practice, target comes from the GMM refined pseudo-labels for that specific patch
            # Mocking target to keep script runnable directly
            pseudo_target = torch.randint(0, num_classes, (data.size(0),)).to(device)

            optimizer.zero_grad()
            logits = model(data)
            loss = F.cross_entropy(logits, pseudo_target)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch [{epoch + 1}/{epochs}] Loss: {epoch_loss / len(train_loader):.4f}"
            )

    # Evaluation Logic
    print("--- Evaluation ---")
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            logits = model(data)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    oa = accuracy_score(all_targets, all_preds)
    cm = confusion_matrix(all_targets, all_preds)
    per_class_acc = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
    aa = np.mean(per_class_acc)
    kappa = cohen_kappa_score(all_targets, all_preds)

    print(f"Overall Accuracy (OA): {oa * 100:.2f}%")
    print(f"Average Accuracy (AA): {aa * 100:.2f}%")
    print(f"Cohen's Kappa (K):     {kappa:.4f}")

    return model, oa, aa, kappa


# Execution trigger (Assuming train_loader, test_loader, and raw_indian_pines_cube exist in your workspace)
# model, oa, aa, kappa = train_special_pipeline(train_loader, test_loader, raw_indian_pines_cube, IP_LABEL_LIST)
